# Notebook 4. Renewal processes, waiting at a bus stop

This notebook follows Notes 4 of IE 8532. The goal is to check the renewal reward theorem, the residual life result, and the inspection paradox by simulation, using buses arriving at a stop.

**Setting.** Buses arrive at a stop with iid headways $X$ and mean headway $\mu = 10$ minutes, so the arrivals form a renewal process. A passenger arrives at a random time far from the start.

**Contents**

1. Build the renewal process in the time frame
2. Part A. Expected wait of a random passenger
3. Part B. The inspection paradox
4. Part C. Renewal reward, cost per minute
5. Summary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": 0.25, "font.size": 10})
rng = np.random.default_rng(8532)

MU = 10.0   # mean headway in minutes

# Three headway distributions, all with mean 10 minutes. Each takes (n, rng) and returns n headways.
headway = {
    "Deterministic 10":     lambda n, rng: np.full(n, 10.0),
    "Exponential, mean 10": lambda n, rng: rng.exponential(10.0, size=n),
    "Uniform(5, 15)":       lambda n, rng: rng.uniform(5.0, 15.0, size=n),
}

## 1. Build the renewal process in the time frame

The arrival times are $S_n = X_1 + \dots + X_n$ (Definition 1). Draw many iid headways and take the cumulative sum, the same way `simulate_pp` in Notebook 1 built a Poisson process from exponential gaps.

**Hints**

- Draw enough headways to cover your time horizon $T$. If $T = 10^6$ minutes and $\mu = 10$, you need a bit more than $T/\mu = 10^5$ headways.
- Return both the headways and the arrival times, since Parts B and C need the headways.

In [ ]:
def simulate_renewal(sampler, T, rng):
    """Return (headways X, arrival times S) for a renewal process run up to at least time T."""
    # TODO 1. Draw about T/MU * 1.2 headways with sampler(n, rng).
    # TODO 2. Take the cumulative sum to get the arrival times S.
    # TODO 3. If S[-1] < T, draw more headways and extend (or just draw more at the start).
    raise NotImplementedError

In [ ]:
# Sanity check. The long-run rate N(t)/t should be close to 1/mu = 0.1 (Strong Law for RPs, Proposition 2).
T = 1_000_000
for name, sampler in headway.items():
    X, S = simulate_renewal(sampler, T, rng)
    # TODO 4. Count the arrivals up to T and print N(T)/T next to 1/MU.

## 2. Part A. Expected wait of a random passenger

The wait of a passenger who arrives at time $t$ is the residual life $Y(t) = S_{N(t)+1} - t$. The notes (slide 42, Example 12) show that

$$\lim_{t\to\infty} E[Y(t)] = \frac{E[X^2]}{2\mu}.$$

**Task.** Simulate passengers, estimate the long-run average wait for each of the three distributions, and compare it with $E[X^2]/(2\mu)$.

**Hints**

- Draw passenger times uniformly from a window well past the start, for example from $0.1T$ to $T$, so the start-up effect is gone.
- `np.searchsorted(S, t)` gives the index of the first arrival after $t$. The wait is `S[idx] - t`.
- For the exact value, use $E[X^2] = \text{Var}(X) + \mu^2$. The three variances are 0, 100, and $100/12$.

In [ ]:
def passenger_waits(S, n_pass, t_lo, t_hi, rng):
    """Waits of n_pass passengers arriving uniformly in (t_lo, t_hi). Also return the index of the next bus."""
    # TODO 5. Draw passenger times, find idx with np.searchsorted, compute the waits.
    raise NotImplementedError

EX2 = {"Deterministic 10": 100.0, "Exponential, mean 10": 200.0, "Uniform(5, 15)": 100.0 + 100.0/12}

# TODO 6. For each distribution, estimate the mean wait and print it next to EX2[name] / (2 * MU).
#         Expected answers are 5.00, 10.00, and 5.42.

## 3. Part B. The inspection paradox

The headway that contains the passenger, $X_{N(t)+1}$, is longer on average than an ordinary headway (slide 28). Its long-run mean is $E[X^2]/\mu$.

**Task.** For each distribution, estimate the average length of the headway the passenger lands in, and compare it with the mean headway of 10 and with $E[X^2]/\mu$.

**Hints**

- Using the same `idx` as in Part A, the containing headway is `S[idx] - S[idx-1]`, or equivalently `X[idx]` if you index the headways the same way.
- Expected answers are 10.00, 20.00, and 10.83.
- Watch the exponential case. The average wait equals the full mean headway of 10, not half of it. Write one sentence in the cell below saying why long headways catch more passengers.

In [ ]:
# TODO 7. Compute and print the mean containing headway for each distribution next to EX2[name] / MU.

In [ ]:
# TODO 8. Plot idea. For the exponential case, a histogram of the simulated waits with the limiting
# density overlaid. The limiting cdf is (integral from 0 to x of (1 - F(y)) dy) / MU (slide 48).

## 4. Part C. Renewal reward, cost per minute

Each cycle between two arrivals earns a reward $R_n$ that may depend on the headway $X_n$. The renewal reward theorem (Proposition 4) says that

$$\lim_{t\to\infty} \frac{R(t)}{t} = \frac{E[R_i]}{\mu}.$$

**Task.** Use the crowding penalty $R = X^2$. Show by simulation that the total reward per minute is close to $E[X^2]/\mu$.

**Hints**

- Pair each reward with the headway that produced it. Here $R_n = X_n^2$.
- Compute the total reward of the cycles completed by time $T$, divide by $T$, and compare it with `mean(R) / mean(X)`.
- Optional extension. Make the reward $R = K + c\,X$ for a fixed trip cost $K$ and a cost $c$ per minute, and compare with $K/\mu + c$.

In [ ]:
# TODO 9. For each distribution, compute R(T)/T and E[R]/mu, and print them side by side.

## 5. Summary

Fill this in when the checks agree with the theory.

| Headway | Mean wait (simulated) | $E[X^2]/(2\mu)$ | Mean containing headway (simulated) | $E[X^2]/\mu$ | $R(T)/T$ | $E[R]/\mu$ |
|---|---|---|---|---|---|---|
| Deterministic 10 | | 5.00 | | 10.00 | | |
| Exponential, mean 10 | | 10.00 | | 20.00 | | |
| Uniform(5, 15) | | 5.42 | | 10.83 | | |

**What I learned.** Write two or three sentences. Why does the same average headway give different waits, and what does that say about bus bunching?